# 04 — Feature Engineering

**PrioMed-Rec | Sub-Task 4**

Builds the 16-feature matrix from admission cohort, medication timelines, and conditions.
Output: `data/processed/feature_matrix.csv`

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.data.admission_cohort import build_admission_cohort
from src.data.medication_timeline import (
    build_medication_timeline, build_pre_admission_meds, build_admission_meds
)
from src.data.loaders import load_conditions, load_encounters
from src.features.feature_pipeline import (
    build_feature_matrix, save_feature_matrix, update_feature_config, FEATURE_VERSION
)
print('Feature version:', FEATURE_VERSION)

## 1. Load interim data

In [ ]:
cohort    = build_admission_cohort('../data/raw/synthea')
timeline  = build_medication_timeline('../data/raw/synthea')
pre_meds  = build_pre_admission_meds(cohort, timeline)
adm_meds  = build_admission_meds(cohort, timeline)
conditions = load_conditions('../data/raw/synthea')
all_enc   = load_encounters('../data/raw/synthea')
print(f'Cohort: {len(cohort):,} | Pre-meds: {len(pre_meds):,} | Adm-meds: {len(adm_meds):,}')

## 2. Build feature matrix

In [ ]:
X = build_feature_matrix(cohort, pre_meds, adm_meds, conditions, all_enc,
                          high_alert_config='../configs/high_alert_medications.json')
print(X.shape)
X.head()

## 3. Feature statistics

In [ ]:
X.describe().round(3)

## 4. Non-zero rates

In [ ]:
nonzero_rates = (X != 0).mean().round(3) * 100
print('Non-zero rate (%) per feature:')
print(nonzero_rates.to_string())

## 5. Polypharmacy & high-alert distribution

In [ ]:
print(f'Polypharmacy rate: {X["polypharmacy_flag"].mean()*100:.1f}%')
print(f'High-alert rate:   {X["high_alert_flag"].mean()*100:.1f}%')
print(f'Mean active conditions: {X["active_condition_count"].mean():.1f}')

## 6. Save feature matrix

In [ ]:
save_feature_matrix(X, '../data/processed/feature_matrix.csv')
update_feature_config('../configs/feature_config.json')
print('Done.')